Notes: 

cell builder is currently embedded in the simulation module... Need to build the 

Also want to implement new clustering method...

In [1]:
import sys
sys.path.append('..')
sys.path.append('../Modules')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(os.getcwd())
if 'notebook' in os.getcwd():
    # os.chdir("../scripts") # go from Neural-Modeling/notebooks to Neural-Modeling/scripts, where simulation outputs will be generated to. (maybe a separate folder could be used...)
    os.chdir("../simulations") # go to output folder
    print(os.getcwd())

/home/drfrbc/Neural-Modeling/notebooks
/home/drfrbc/Neural-Modeling/simulations


User Specifications -  get parameters, simulation folder

In [2]:
from Modules.constants import HayParameters
import datetime
import pickle

sim_set_title = "description_of_simulation_set"
# create simulation folder
sims_dir = f"{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M')}-{sim_set_title}"
os.makedirs(sims_dir, exist_ok=True)

# this is a quick example, but the generation of parameter sets and titles is in scripts/gen_param_list_advanced.py
sim_titles = ["description_of_simulation"]
parameter_sets = [HayParameters(sim_title) for sim_title in sim_titles] # example parameter set

# create simulation folders and save parameters in them
for parameters, sim_title in zip(parameter_sets, sim_titles):
    # create simulation folder
    sim_dir = os.path.join(sims_dir, sim_title)
    os.makedirs(sim_dir, exist_ok=True)

    with open(os.path.join(sim_dir, "params.pickle"), 'wb') as file:
        pickle.dump(parameters, file)

# @TODO: at simulation time, the parameters should be loaded from each sim_dir in sims_dir. 
# We can make one script that does this code snippet, then passes sims_dir to the mpiexec simulation script.

--No graphics will be displayed.


Build cell to save segments csv

In [ ]:
from Modules.cell_builder import CellBuilder
from Modules.logger import Logger
from Modules.cell_builder import SkeletonCell
# this script would be called with mpiexec

# @TODO: would we want to use mpiexec for this script? It depends on if there are different morphologies: different cell types, different reduction parameters, and segmentation. Yes, one per simulation folder. might as well.
# Additionally, do we save these in simulation folders or try to use a common folder? A: simulation folder

# for rank in [0]: # replace with actual mpi implementation

skeleton_cell_type = 'Hay' # @TODO: use a parameter for skeleton_cell_type.

for sim_dir in os.listdir(sims_dir):
    sim_dir = os.path.join(sims_dir, sim_dir)
    if not os.path.isdir(sim_dir):
        continue

    # load parameters
    with open(os.path.join(sim_dir, "params.pickle"), 'rb') as file:
        parameters = pickle.load(file)

    logger = Logger(sim_dir)

    # build the cell
    cell_builder = CellBuilder(getattr(SkeletonCell, skeleton_cell_type), parameters, logger)
    cell, _ = cell_builder.build_cell()

    # manipulate morphology: reduction, segmentation
    #@TODO code from cellbuilder.py: CellBuilder.build_cell  -lines around reductor code block

    # save segments csv in simulation folder - the rest of this cell
    #@TODO: Make modularized code for this and clean. standardize between here and cell_model (this is from simulation.py)
    if parameters.save_adj_matrix:
        adj_matrix = cell.compute_directed_adjacency_matrix()
        np.savetxt(os.path.join(parameters.path, "adj_matrix.txt"), adj_matrix)

    # Classify segments by morphology, save coordinates
    segments, seg_data = cell.get_segments(["all"]) # (segments is returned here to preserve NEURON references)
    seg_sections = []
    seg_idx = []
    seg_coords = []
    seg_half_seg_RAs = []
    seg = []
    seg_Ls = []
    sec_Ls = []
    sec_Ds = []
    seg_distance = []
    psegs=[]
    
    for i,entry in enumerate(seg_data):
        # if parameters.build_stylized: #@DEPRACATED
        #     sec_name = entry.section.split(".")[-1]
        # else:
        sec_name = entry.section.split(".")[-1] # name[idx]
        #print(f"sec_name: {sec_name}")
        seg_sections.append(sec_name.split("[")[0])
        seg_idx.append(sec_name.split("[")[1].split("]")[0])
        seg_coords.append(entry.coords)
        seg_half_seg_RAs.append(entry.seg_half_seg_RA)
        seg.append(entry.seg)
        seg_Ls.append(entry.L)
        psegs.append(entry.pseg)
        sec_Ls.append(segments[i].sec.L)
        sec_Ds.append(segments[i].sec.diam)
        seg_distance.append(h.distance(segments[0], segments[i]))
        
        
    seg_sections = pd.DataFrame({
        "section": seg_sections, 
        "idx_in_section_type": seg_idx,
        "seg_half_seg_RA": seg_half_seg_RAs,
        "L": seg_Ls,
        "seg":seg,
        "pseg":psegs,
        "Section_L":sec_Ls,
        "Section_diam":sec_Ds,
        "Distance":seg_distance
        })

    seg_coords = pd.concat(seg_coords)

    seg_data = pd.concat((seg_sections.reset_index(drop = True), seg_coords.reset_index(drop = True)), axis = 1)
    seg_data.to_csv(os.path.join(parameters.path, "segment_data.csv"))


NameError: name 'self' is not defined

Generate synapse Locations - load segments csv

In [ ]:
import pandas as pd
from functools import partial
# define a class for generating synapses abstractly
class PreSimSynapseGenerator:
    def __init__(self, segments, parameters):
        self.segments = segments
        self.parameters = parameters
        self.synapses = pd.DataFrame()

        if self.parameters.segment_measurement_for_probabilities not in ['length', 'surface_area']:
            raise ValueError(f"Measurement for probabilities must be 'length' or 'surface_area'. Not {self.parameters.segment_measurement_for_probabilities}.")

    def generate_synapse_locations(self):
        for sec_type, synapse_properties in self.parameters.inh_syn_properties.items():
            segments_to_generate_on = self.get_segments_of_type(sec_type, self.segments) # get segments
            self.random_state = np.random.RandomState(self.parameters.inh_syn_properties[sec_type]['seed']['synapses'])
            np.random.seed(self.parameters.inh_syn_properties[sec_type]['seed']['synapses'])

            self.build_synapses_with_specs(segments_to_generate_on = segments_to_generate_on,
                                      sec_type = sec_type,
                                      synapse_type = synapse_properties['synapse_type'],
                                      use_density = self.parameters.use_density,
                                      syn_number = synapse_properties['syn_number'] if not self.parameters.use_density else None,
                                      syn_density = synapse_properties['syn_density'] if self.parameters.use_density else None,
                                      initial_weight_distribution = synapse_properties['initial_weight_distribution'],
                                      release_probability_distribution = synapse_properties['release_probability_distribution'],
                                      name = f"inh_{sec_type}"
                                      )


        pass
    def generate_synapse_spike_trains(self):
        pass
    def generate_synapse_weights(self):
        pass

    def get_segments_of_type(self, sec_type, segments):
        return segments[segments['sec_type'] == sec_type]
    
    def build_synapses_with_specs(self, segments_to_generate_on: pd.DataFrame, sec_type: str, synapse_type: str, use_density: bool,
                                  syn_number: int, syn_density: float, initial_weight_distribution: dict,
                                  release_probability_distribution: dict, name: str):
        
        if initial_weight_distribution is None or release_probability_distribution is None:
            raise ValueError("Both gmax_dist_params and P_release_params must be provided.")
        
        if syn_number is None and syn_density is None:
            raise ValueError("Either syn_number or syn_density must be provided.")
        elif syn_number is not None and syn_density is not None:
            raise ValueError("Only one of syn_number or syn_density must be provided.")
        elif syn_number is not None and use_density:
            raise ValueError("syn_number should be none when using density.")
        elif syn_density is not None and not use_density:
            raise ValueError("syn_density should be none when not using density.")

        if initial_weight_distribution['function'] is not None:
            initial_weight_distribution = partial(initial_weight_distribution['function'], **initial_weight_distribution['params'], size=1)
        else:
            initial_weight_distribution = initial_weight_distribution['params']['mean'] # use mean if no function is provided
        self.logger.log(f"Initial weight distribution for {sec_type}: {initial_weight_distribution}")

        if release_probability_distribution['function'] is not None:
            release_probability_distribution = partial(release_probability_distribution['function'], **release_probability_distribution['params'], size=1)
        else:
            release_probability_distribution = release_probability_distribution['params']['mean']
        self.logger.log(f"Release probability distribution for {sec_type}: {release_probability_distribution}")

        # calculate probabilities of placing synapses on segments
        total_measurement = segments_to_generate_on[self.parameters.segment_measurement_for_probabilities].sum()
        self.logger.log(f"Total {self.parameters.segment_measurement_for_probabilities} for {sec_type}: {total_measurement}")
        segments_to_generate_on['probability'] = segments_to_generate_on[self.parameters.segment_measurement_for_probabilities] / total_measurement

        if segments_to_generate_on['probability'].sum() != 1:
            raise ValueError("Probabilities do not sum to 1. Check your segment measurement for probabilities.")

        #@TODO: make sure that within 50 microns is excluded from non-perisomatic types when generating segments.csv

        if use_density:
            # calculate number of synapses per segment
            syn_number = total_measurement * syn_density #@TODO make compatible with syn_density being function instead of float (not at all urgent)
            self.logger.log(f"Total number of synapses for {sec_type}: {syn_number}")

        


In [ ]:
# load segments csv
pd.read_csv(os.path.join(parameters.path, "segment_data.csv"))

## generate synapse locations - code from cellbuilder @TODO: need to change to utilize segment_data.csv instead of cell_model.py

#

# save to synapses csv in simulation folder # example in cell_builder of getting properties

Generate synapse weights

In [ ]:
# load synapse locations from synapses csv

# generate synapse weights, params, etc based on locations

# save to synapses csv in simulation folder

Generate Spike Trains

In [ ]:
# generate functional groups from params

# generate presynaptic cells from functional groups and params

# generate spike trains for presynaptic cells from params

# load synapse locations from synapses csv

# cluster synapse locations into presynaptic cells

# assign synapses to presynaptic cells

# save spike trains and synapse assignments as spike_trains.csv to simulation folder

Read synapse specifications and build cell with synapses

In [ ]:
# build synapses from synapses csv from simulation folder

# set spike trains from spike_trains.csv from simulation folder

